# Rutas dentro del notebook

In [1]:
import sys
import os

# Esto asegura que trabajes desde la raíz
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Conexión a SQL Server

In [2]:
# Cargar variables desde .env

from dotenv import load_dotenv
import os

# Cargar .env desde la raíz del proyecto
load_dotenv(os.path.join(BASE_DIR, ".env"))

DB_SERVER = os.getenv("DB_SERVER")
DB_DATABASE = os.getenv("DB_DATABASE")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

print("Variables cargadas correctamente")

Variables cargadas correctamente


In [3]:
# Conectar a SQL Server
import pyodbc

try:
    conn = pyodbc.connect(
        f"DRIVER={{SQL Server}};"
        f"SERVER={DB_SERVER};"
        f"DATABASE={DB_DATABASE};"
        f"UID={DB_USER};"
        f"PWD={DB_PASSWORD}"
    )
    
    cursor = conn.cursor()
    print("✅ Conexión exitosa a SQL Server")

except Exception as e:
    print("❌ Error de conexión:", e)

✅ Conexión exitosa a SQL Server


In [4]:
# Verificar conexión (MUY importante)
cursor.execute("SELECT TOP 5 name FROM sys.tables")

for row in cursor.fetchall():
    print(row)

('CARGOS',)
('CARPETA_ACT_EXT',)
('CARPETA_ACT_INT',)
('TIPO_DOCUMENTO',)
('DIST_FISCAL',)


# extracción de la información de la BD - QUERY SQL COMPLETA (JOIN REAL)

In [5]:
query = """
SELECT TOP 10
    nd.DESPACHO,
    nd.RESPONSABLE,
    nd.FECHA,
    nd.COD_NUM_DOC,
    
    c.AÑOCASO,
    c.NUMCASO,
    c.SECCUENCIACASO,

    l.DOCUMENTO,
    l.EXT_DOC_CASO

FROM NDFPCYFCH.dbo.NUMERODOC nd

INNER JOIN NDFPCYFCH.dbo.CASO c
    ON nd.DESPACHO = c.CODDESPACHO
    AND nd.COD_NUM_DOC = c.CODNUMDOC

INNER JOIN NDFPCYFCH.dbo.LEGADOCCASO l
    ON nd.DESPACHO = l.DESPACHO
    AND nd.COD_NUM_DOC = l.COD_NUM_DOC

WHERE nd.TIPO_DOCUMENTO = 2

ORDER BY nd.FECHA DESC
"""

Extraer la data - EJECUTAR Y EXTRAER

In [6]:
cursor.execute(query)
rows = cursor.fetchall()

print(f"Total registros: {len(rows)}")

Total registros: 10


GUARDAR ARCHIVOS (LO MÁS IMPORTANTE)


In [7]:
import os

for row in rows:
    despacho = str(row.DESPACHO)
    responsable = str(row.RESPONSABLE).replace(" ", "_")
    anio = str(row.AÑOCASO)
    numcaso = str(row.NUMCASO)
    secuencia = str(row.SECCUENCIACASO)

    # 🔧 Manejo robusto de la extensión
    extension = (row.EXT_DOC_CASO or "bin").strip().lower()

    if extension == "":
        extension = "bin"

    if not extension.startswith("."):
        extension = "." + extension

    # 📄 Nombre del archivo
    nombre_archivo = f"{despacho}_{responsable}_{anio}_{numcaso}_{secuencia}{extension}"

    # 📂 Ruta final
    file_path = os.path.join(RAW_DIR, nombre_archivo)

    try:
        # 💾 Guardar archivo binario
        with open(file_path, "wb") as f:
            f.write(row.DOCUMENTO)

        print(f"✅ Guardado: {nombre_archivo}")

    except Exception as e:
        print(f"❌ Error con {nombre_archivo}: {e}")

✅ Guardado: 6411_430785016411_______2026_50_0.doc
✅ Guardado: 6411_074650786411_______2026_49_0.doc
✅ Guardado: 6411_181623036411_______2026_47_0.doc
✅ Guardado: 6411_181623036411_______2026_46_0.doc
✅ Guardado: 6411_181623036411_______2026_44_0.doc
✅ Guardado: 6411_074650786411_______2026_41_0.doc
✅ Guardado: 6411_074650786411_______2026_40_0.doc
✅ Guardado: 6411_181623036411_______2026_42_0.doc
✅ Guardado: 6411_181623036411_______2026_38_0.doc
✅ Guardado: 6411_074650786411_______2026_34_0.doc


# FASE 7 — Detección + Conversión AUTOMÁTICA 

In [8]:
import subprocess
import os

def convertir_a_pdf(input_path, output_dir):
    try:
        subprocess.run([
            r"C:\Program Files\LibreOffice\program\soffice.exe",
            "--headless",
            "--convert-to", "pdf",
            input_path,
            "--outdir", output_dir
        ], check=True)

        print(f"🔄 Convertido: {os.path.basename(input_path)}")

    except FileNotFoundError:
        print("❌ LibreOffice no encontrado. Verifica la ruta.")

    except subprocess.CalledProcessError as e:
        print(f"❌ Error al convertir {input_path}: {e}")

SIGUIENTE PASO — PROCESAR TODOS LOS ARCHIVOS

In [9]:
for file in os.listdir(RAW_DIR):
    ruta = os.path.join(RAW_DIR, file)
    
    print(f"Procesando: {file}")
    convertir_a_pdf(ruta, PROCESSED_DIR)

Procesando: 6411_074650786411_______2026_34_0.doc
🔄 Convertido: 6411_074650786411_______2026_34_0.doc
Procesando: 6411_074650786411_______2026_40_0.doc
🔄 Convertido: 6411_074650786411_______2026_40_0.doc
Procesando: 6411_074650786411_______2026_41_0.doc
🔄 Convertido: 6411_074650786411_______2026_41_0.doc
Procesando: 6411_074650786411_______2026_49_0.doc
🔄 Convertido: 6411_074650786411_______2026_49_0.doc
Procesando: 6411_181623036411_______2026_38_0.doc
🔄 Convertido: 6411_181623036411_______2026_38_0.doc
Procesando: 6411_181623036411_______2026_42_0.doc
🔄 Convertido: 6411_181623036411_______2026_42_0.doc
Procesando: 6411_181623036411_______2026_44_0.doc
🔄 Convertido: 6411_181623036411_______2026_44_0.doc
Procesando: 6411_181623036411_______2026_46_0.doc
🔄 Convertido: 6411_181623036411_______2026_46_0.doc
Procesando: 6411_181623036411_______2026_47_0.doc
🔄 Convertido: 6411_181623036411_______2026_47_0.doc
Procesando: 6411_430785016411_______2026_50_0.doc
🔄 Convertido: 6411_430785016411_

# FASE 8 — Subir a Azure Data Lake

In [10]:
from azure.storage.blob import BlobServiceClient
import os

conn_str = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "landing-zone"

blob_service_client = BlobServiceClient.from_connection_string(conn_str)

for file in os.listdir(PROCESSED_DIR):
    file_path = os.path.join(PROCESSED_DIR, file)

    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=file
    )

    with open(file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)

    print(f"☁️ Subido: {file}")

☁️ Subido: 6411_074650786411_______2026_34_0.pdf
☁️ Subido: 6411_074650786411_______2026_40_0.pdf
☁️ Subido: 6411_074650786411_______2026_41_0.pdf
☁️ Subido: 6411_074650786411_______2026_49_0.pdf
☁️ Subido: 6411_181623036411_______2026_38_0.pdf
☁️ Subido: 6411_181623036411_______2026_42_0.pdf
☁️ Subido: 6411_181623036411_______2026_44_0.pdf
☁️ Subido: 6411_181623036411_______2026_46_0.pdf
☁️ Subido: 6411_181623036411_______2026_47_0.pdf
☁️ Subido: 6411_430785016411_______2026_50_0.pdf


# FASE 9 — Procesar con IA

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

True

# FASE FINAL — EXTRACCIÓN INTELIGENTE (TODO EN UNO)

PASO 1 — FUNCIÓN PRINCIPAL DE EXTRACCIÓN

In [12]:
import re
from datetime import datetime, timedelta

def extraer_datos(texto):
    texto = texto.lower()
    
    # -------- CASO --------
    caso = re.search(r"caso.*n[°º]\s*([\w\-]+)", texto)
    caso = caso.group(1) if caso else "NaN"
    
    # -------- FECHA --------
    fecha_match = re.search(r"\d{1,2} de \w+ de \d{4}", texto)
    fecha_str = fecha_match.group(0) if fecha_match else None
    
    # -------- PLAZO --------
    plazo_match = re.search(r"(\d+)\s+d[ií]as", texto)
    plazo = int(plazo_match.group(1)) if plazo_match else None
    
    # -------- DELITO --------
    delitos = ["violencia", "abandono", "maltrato", "riesgo", "explotación", "contravención"]
    delito = next((d for d in delitos if d in texto), "NaN")
    
    # -------- PARTES --------
    def buscar(patron):
        m = re.search(patron, texto)
        return m.group(1).strip() if m else "NaN"
    
    agraviado = buscar(r"agraviad[oa]:?\s*([a-z\s]+)")
    investigado = buscar(r"investigad[oa]:?\s*([a-z\s]+)")
    
    # -------- INICIALES FISCAL --------
    fiscal = re.search(r"\b[A-Z]{2,4}\/\.", texto.upper())
    fiscal = fiscal.group(0) if fiscal else "NaN"
    
    # -------- EDAD --------
    edad = re.search(r"(\d{1,2})\s+años", texto)
    edad = int(edad.group(1)) if edad else "NaN"
    
    # -------- CONVERSIÓN FECHA --------
    meses = {
        "enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,
        "julio":7,"agosto":8,"septiembre":9,"octubre":10,"noviembre":11,"diciembre":12
    }
    
    fecha_inicio = None
    if fecha_str:
        try:
            partes = fecha_str.split()
            dia = int(partes[0])
            mes = meses[partes[2]]
            anio = int(partes[4])
            fecha_inicio = datetime(anio, mes, dia)
        except:
            pass
    
    # -------- ESTADO --------
    estado = "NaN"
    if fecha_inicio and plazo:
        vencimiento = fecha_inicio + timedelta(days=plazo)
        hoy = datetime.now()
        
        if hoy > vencimiento:
            estado = "VENCIDO"
        elif (vencimiento - hoy).days <= 5:
            estado = "POR VENCER"
        else:
            estado = "VIGENTE"
    
    return {
        "caso": caso,
        "fecha_inicio": fecha_str if fecha_str else "NaN",
        "plazo_dias": plazo if plazo else "NaN",
        "estado": estado,
        "delito": delito,
        "agraviado": agraviado,
        "investigado": investigado,
        "edad": edad,
        "fiscal": fiscal
    }

Antes de ejecutar paso dos — importamos librerias y cargasmos el endpoint y credential

In [13]:
import os
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

client = DocumentAnalysisClient(
    endpoint=os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("DOCUMENT_INTELLIGENCE_KEY"))
)

PASO 2 — PROCESAR TODOS LOS PDFs

In [14]:
import pandas as pd
import os

resultados = []

for file in os.listdir(PROCESSED_DIR):
    ruta = os.path.join(PROCESSED_DIR, file)

    try:
        with open(ruta, "rb") as f:
            poller = client.begin_analyze_document(
                "prebuilt-document",
                document=f
            )
            result = poller.result()

        texto = result.content

        data = extraer_datos(texto)
        data["archivo"] = file

        resultados.append(data)

    except Exception as e:
        print(f"❌ Error con {file}: {e}")

df = pd.DataFrame(resultados)

df.head()

,caso,fecha_inicio,plazo_dias,estado,delito,agraviado,investigado,edad,fiscal,archivo
0,1706054800-2026-34-0,26 de febrero de 2026,NaN,NaN,riesgo,NaN,NaN,18,NaN,6411_074650786411_______2026_34_0.pdf
1,NaN,06 de marzo de 2026,NaN,NaN,NaN,NaN,NaN,7,NaN,6411_074650786411_______2026_40_0.pdf
2,NaN,09 de marzo de 2026,NaN,NaN,NaN,s y dentro de los l,s,7,NaN,6411_074650786411_______2026_41_0.pdf
3,1706054800-2026-49-0,11 de marzo de 2026,NaN,NaN,NaN,NaN,a la persona de anair aracely vallejos julca e...,NaN,NaN,6411_074650786411_______2026_49_0.pdf
4,NaN,19 de marzo de 2026,NaN,NaN,NaN,s y dentro de los l,s,15,NaN,6411_181623036411_______2026_38_0.pdf


# OBJETIVO FINAL 

PASO 1 — Función de cálculo

In [15]:
from datetime import datetime, timedelta
import re

def calcular_estado(data):
    
    texto = data.get("texto", "")
    
    # 📅 1. Extraer fecha (ej: "cuatro de marzo del año dos mil veintiséis")
    fecha = None
    
    match_fecha = re.search(r"(\d{1,2} de [a-z]+ del año \d{4})", texto.lower())
    
    if match_fecha:
        try:
            fecha = datetime.strptime(match_fecha.group(1), "%d de %B del año %Y")
        except:
            fecha = None

    # ⏳ 2. Extraer días (ej: "por el plazo de 60 días")
    dias = None
    match_dias = re.search(r"(\d+)\s*d[ií]as", texto.lower())
    
    if match_dias:
        dias = int(match_dias.group(1))

    # 📆 3. Calcular vencimiento
    fecha_vencimiento = None
    estado = "NAN"

    if fecha and dias:
        fecha_vencimiento = fecha + timedelta(days=dias)
        hoy = datetime.today()

        if hoy > fecha_vencimiento:
            estado = "VENCIDO"
        elif (fecha_vencimiento - hoy).days <= 5:
            estado = "POR VENCER"
        else:
            estado = "EN PLAZO"

    return {
        "fecha_resolucion": fecha,
        "dias_investigacion": dias,
        "fecha_vencimiento": fecha_vencimiento,
        "estado": estado
    }

PASO 2 — Integrarlo a tu pipeline

In [16]:
resultados = []

for file in os.listdir(PROCESSED_DIR):
    ruta = os.path.join(PROCESSED_DIR, file)

    try:
        with open(ruta, "rb") as f:
            poller = client.begin_analyze_document(
                "prebuilt-document",
                document=f
            )
            result = poller.result()

        texto = result.content

        data = extraer_datos(texto)
        data["texto"] = texto  # 👈 IMPORTANTE
        data["archivo"] = file

        # 🔥 NUEVO
        calculo = calcular_estado(data)
        data.update(calculo)

        resultados.append(data)

    except Exception as e:
        print(f"❌ Error con {file}: {e}")

PASO 3 — Crear dataset limpio

In [17]:
df = pd.DataFrame(resultados)

df.head()

,caso,fecha_inicio,plazo_dias,estado,delito,agraviado,investigado,edad,fiscal,texto,archivo,fecha_resolucion,dias_investigacion,fecha_vencimiento
0,1706054800-2026-34-0,26 de febrero de 2026,NaN,NAN,riesgo,NaN,NaN,18,NaN,DISTRITO FISCAL DE CAJAMARCA FISCALIA PROVINCI...,6411_074650786411_______2026_34_0.pdf,None,NaN,None
1,NaN,06 de marzo de 2026,NaN,NAN,NaN,NaN,NaN,7,NaN,MINISTERIO PÚBLICO FISCALÍA DE LA NACIÓN\nDIST...,6411_074650786411_______2026_40_0.pdf,None,NaN,None
2,NaN,09 de marzo de 2026,NaN,NAN,NaN,s y dentro de los l,s,7,NaN,MINISTERIO PÚBLICO FISCALÍA DE LA NACIÓN\nDIST...,6411_074650786411_______2026_41_0.pdf,None,NaN,None
3,1706054800-2026-49-0,11 de marzo de 2026,NaN,NAN,NaN,NaN,a la persona de anair aracely vallejos julca e...,NaN,NaN,MINISTERIO PÚBLICO FISCALÍA DE LA NACIÓN\nCASO...,6411_074650786411_______2026_49_0.pdf,None,NaN,None
4,NaN,19 de marzo de 2026,NaN,NAN,NaN,s y dentro de los l,s,15,NaN,MINISTERIO PÚBLICO FISCALÍA DE LA NACIÓN\nDIST...,6411_181623036411_______2026_38_0.pdf,None,NaN,None


PASO 4 — Exportar (PROFESIONAL)

In [21]:
# CSV
df.to_csv("data_final.csv", index=False)

# JSON
df.to_json("data_final.json", orient="records", force_ascii=False)

# PASO AWS — subir tu dataset final

Celda final extra (MULTICLOUD)

Falta realizar por falta de tiempo. ¡Pero, ha sido un camino facinante!